In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch
import copy

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer.IntervalTrainer import IntervalTrainer
from src.trainer.SmoothTrainer import SmoothTrainer
from src.trainer.SimpleTrainer import SimpleTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets
from src.utils import InContextHead
from src import models
from src.regulariser.UnbiasRegulariser import UnbiasRegulariser
from src.regulariser.MultiRegulariser import MultiRegulariser
from src.regulariser.L2Regulariser import L2Regulariser

/vol/bitbucket/le24/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Task Incremental Learning

In [4]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cpu")
head.set_context(0)
model = models.get_fully_connected_mnist_model(head=head, dense_layers=3)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


Train the model on the original task.

In [5]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=5, batch_size=256)
trainer.test(test_tasks[0:1])


Training Epochs:   0%|                                                                                                  | 0/5 [00:00<?, ?it/s, val_loss=N\A]

Training Epochs: 100%|███████████████████████████████████████████████████████████████████████| 5/5 [00:20<00:00,  4.06s/it, val_loss=0.00822, val_acc=0.997]


Test Results: [(0.0079, 0.9976)]


In [6]:
# LORA BLOCK
from peft import LoraConfig
from peft.tuners.lora import LoraModel
from torch.utils.data import DataLoader

# Create your config
lora_config = LoraConfig(
    r=2,
    lora_alpha=4,
    target_modules=["1", "3", "5", "7"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"  # as a string for manual use
)

# Wrap manually with LoraModel
peft_model = LoraModel(trainer.model, lora_config, adapter_name="default") # ? Why not just do this in SmoothTrainer directly?

buffer_subset = torch.utils.data.Subset(train_tasks[0], list(range(1, 2500)))
buffer_loader = DataLoader(buffer_subset, batch_size=256, shuffle=True)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(peft_model.parameters(), lr=1e-3)

peft_model.train()
total_loss = 0

for epoch in range(5):
    for x, y in buffer_loader:
    
        # Optionally set context here
        if isinstance(peft_model.model[-1], InContextHead):
            peft_model.model[-1].set_context(0)  # replace 0 with actual context_id if needed
    
        optimizer.zero_grad()
        out = peft_model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
    
        total_loss += loss.item()
    
    print(f"Total loss for epoch: {total_loss:.4f}")


Total loss for epoch: 0.0212
Total loss for epoch: 0.0397
Total loss for epoch: 0.0561
Total loss for epoch: 0.0712
Total loss for epoch: 0.0845


In [7]:
correct = 0
total = 0

if isinstance(peft_model.model[-1], InContextHead):
    peft_model.model[-1].set_context(0) 

val_loader = DataLoader(val_tasks[0], batch_size=256, shuffle=True)
with torch.no_grad():
    for x, y in val_loader:
        outputs = peft_model(x)
        preds = outputs.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

accuracy = correct / total
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9975


In [8]:

smooth_trainer = SmoothTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=300,
    min_acc_limit=0.95,
    min_acc_increment=0.01,
    primal_learning_rate=0.5,
    paradigm="TIL",
)

smooth_trainer.smooth_cheat = 1_000.0 
smooth_trainer.smooth_delta = 0.1

smooth_trainer.peft_model = peft_model

smooth_trainer.density_strategy = "LoRA"

smooth_trainer.compute_rashomon_set(test_tasks[0], context_id=0)


  0%|                                                                                                                               | 0/100 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:58<00:00,  1.72it/s]


Inflate: 5.0, Estimated success: 1.0000, Certified radius: 20.0658


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:54<00:00,  1.83it/s]


Inflate: 10.0, Estimated success: 1.0000, Certified radius: 40.1315


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:58<00:00,  1.71it/s]

Inflate: 15.0, Estimated success: 0.8900, Certified radius: 18.2677
[LoRA] Max certified radius: 4.0132 at inflate 10


In [9]:
def project_lora_to_l2_ball(peft_model, epsilon, quite=True):
    deltas = []

    # Step 1: Collect deltas across modules
    for module in peft_model.modules():
        if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
            A, B = module.lora_A.default.weight, module.lora_B.default.weight
            A0, B0 = module.A0, module.B0
            deltas.append((A, A0))
            deltas.append((B, B0))

    # Step 2: Compute total norm
    total_sq_norm = sum(((p - p0) ** 2).sum() for (p, p0) in deltas)
    total_norm = total_sq_norm.sqrt()
    if not quite:
        print("Norm:", total_norm.item())
    # Step 3: Project if needed
    if total_norm > epsilon:
        scale = epsilon / total_norm
        for (p, p0) in deltas:
            with torch.no_grad():
                delta = (p - p0).detach()
                p.data = p0 + scale * delta


def attach_initial_lora_state(model):
    for module in model.modules():
        if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
            module.A0 = module.lora_A.default.weight.detach().clone()
            module.B0 = module.lora_B.default.weight.detach().clone()

attach_initial_lora_state(peft_model)

In [10]:
context_id = 1  # TIL context index

peft_model.train()
for epoch in range(20):
    total_loss = 0
    for x, y in DataLoader(train_tasks[context_id], batch_size=256, shuffle=True):
        #x, y = x.to(peft_model.device), y.to(peft_model.device)

        # Set task context
        if isinstance(peft_model.model[-1], InContextHead):
            peft_model.model[-1].set_context(context_id)

        optimizer.zero_grad()
        out = peft_model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        project_lora_to_l2_ball(peft_model, smooth_trainer.radius)

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Epoch 1, Loss: 89.7460
Epoch 2, Loss: 10.2162
Epoch 3, Loss: 7.8225
Epoch 4, Loss: 7.3087
Epoch 5, Loss: 7.1941
Epoch 6, Loss: 7.0897
Epoch 7, Loss: 6.8835
Epoch 8, Loss: 6.8730
Epoch 9, Loss: 6.7209
Epoch 10, Loss: 6.7672
Epoch 11, Loss: 6.8389
Epoch 12, Loss: 6.6881
Epoch 13, Loss: 6.6858
Epoch 14, Loss: 6.3063
Epoch 15, Loss: 6.3102
Epoch 16, Loss: 6.5122
Epoch 17, Loss: 6.3961
Epoch 18, Loss: 6.2714
Epoch 19, Loss: 6.1878
Epoch 20, Loss: 6.2279


In [11]:
correct = 0
total = 0

with torch.no_grad():
    for x, y in  DataLoader(train_tasks[context_id], batch_size=256, shuffle=True):
        
        outputs = peft_model(x)
        preds = outputs.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

accuracy = correct / total
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9879


In [ ]:
prev_peft = copy.deepcopy(smooth_trainer.peft_model)

In [20]:
correct = 0
total = 0

smooth_trainer.peft_model = prev_peft
if isinstance(smooth_trainer.peft_model.model[-1], InContextHead):
    smooth_trainer.peft_model.model[-1].set_context(1)  # replace 1 with actual context_id if needed

with torch.no_grad():
    for x, y in DataLoader(train_tasks[context_id], batch_size=256, shuffle=True):

        # ✅ Sample noisy weights and apply to LoRA modules
        # new_weights = []
        # for loc, scale in zip(smooth_trainer.LoRA_loc, smooth_trainer.LoRA_scale):
        #     noise = torch.normal(mean=0.0, std=0.0*scale).to(loc.device)
        #     new_weights.append(loc + noise)

        # # ✅ Apply perturbed weights
        # idx = 0
        # for module in smooth_trainer.peft_model.modules():
        #     if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
        #         module.lora_A.default.weight.copy_(new_weights[idx])
        #         module.lora_B.default.weight.copy_(new_weights[idx + 1])
        #         idx += 2

        out = smooth_trainer.peft_model(x)
        preds = out.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

accuracy = correct / total
print(f"✅ Accuracy: {accuracy:.4f}")

✅ Accuracy: 0.5005


Train subsequent tasks with bounds.

In [21]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=300,
    min_acc_limit=0.95,
    min_acc_increment=0.01,
    primal_learning_rate=0.5,
    paradigm="TIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0], context_id=0)


---------------------------- Computing Rashomon set ----------------------------


ValueError: Unsupported module type in IBP: <class 'peft.tuners.lora.layer.Linear'>

In [ ]:
import copy

smooth_trainer = SmoothTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=300,
    min_acc_limit=0.95,
    min_acc_increment=0.01,
    primal_learning_rate=0.5,
    paradigm="TIL",
)

smooth_trainer.smooth_cheat = 1_000.0 
smooth_trainer.smooth_delta = 0.1

#smooth_trainer.density_strategy == "interval_init"
#smooth_trainer.pre_computed_bound = interval_trainer.bounds[-1]

smooth_trainer.density_strategy = "fim_init"

#smooth_trainer.smooth_steps = 10
#smooth_trainer.density_strategy = "sgd_opt"


smooth_trainer.compute_rashomon_set(test_tasks[0], context_id=0)

#st_safe = copy.deepcopy(smooth_trainer)


In [ ]:
print(test_tasks[0])

smooth_trainer.test(test_tasks[0:1])


In [ ]:
import copy

# Safely clone everything before test (do this ONCE and store the copies)
orig_model = copy.deepcopy(smooth_trainer.model)
orig_loc = [p.clone() for p in smooth_trainer.loc]
orig_scale = [s.clone() for s in smooth_trainer.scale]
orig_radius = smooth_trainer.radius


In [ ]:

print("TRAINING WITH IBP CONSTRIAINTS: " )

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256, context_id=i)
    interval_trainer.test(test_tasks[0 : i + 1], context_list=list(range(i + 1)))
    break
    #if i < len(train_tasks) - 2:
    #    interval_trainer.compute_rashomon_set(test, context_id=i)


print("TRAINING WITH SMOOTHING CONSTRIAINTS: " )

smooth_trainer.model.load_state_dict(orig_model.state_dict())  # restores weights
smooth_trainer.model.to(smooth_trainer.device)  # ensure correct device
smooth_trainer.loc = [p.clone().to(smooth_trainer.device) for p in orig_loc]
smooth_trainer.scale = [s.clone().to(smooth_trainer.device) for s in orig_scale]
smooth_trainer.radius = 50  # or new radius of your choosing

# FIM Rad of 40+ is very good.
for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    smooth_trainer.train(train, val, batch_size=256, context_id=i)
    smooth_trainer.test(test_tasks[0 : i + 1], context_list=list(range(i + 1)))
    break

 

### Domain Incremental Training

In [ ]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model()

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [ ]:
l2_reg = L2Regulariser(lmbd=0.01)
ubias_reg = UnbiasRegulariser(lmbd=0.01)
regulariser = MultiRegulariser([l2_reg, ubias_reg])

trainer = SimpleTrainer(model, domain_map_fn=domain_map_fn)
trainer.train(
    train_tasks[0],
    val_tasks[0],
    epochs=3,
)
trainer.test(test_tasks[0:1])

In [ ]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.9,
    primal_learning_rate=0.33,
    dual_learning_rate=0.01,
    batch_size=400,
    paradigm="DIL",
    domain_map_fn=domain_map_fn
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])


In [ ]:
import copy

smooth_trainer = SmoothTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.9,
    primal_learning_rate=0.33,
    dual_learning_rate=0.01,
    batch_size=400,
    paradigm="DIL",
    domain_map_fn=domain_map_fn
)

smooth_trainer.smooth_cheat = 1_000.0 
smooth_trainer.smooth_delta = 0.1

#smooth_trainer.density_strategy == "interval_init"
#smooth_trainer.pre_computed_bound = interval_trainer.bounds[-1]

smooth_trainer.density_strategy = "fim_init"

#smooth_trainer.smooth_steps = 10
#smooth_trainer.density_strategy = "sgd_opt"


smooth_trainer.compute_rashomon_set(test_tasks[0])

#st_safe = copy.deepcopy(smooth_trainer)

In [ ]:

print("TRAINING WITH INTERVAL CONSTRAINTS: ")

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:2], val_tasks[1:2], test_tasks[1:2])
):
    interval_trainer.train(train, val, batch_size=256)
    interval_trainer.test(test_tasks[0 : i + 2])
    # if i < len(train_tasks) - 2:
    #     interval_trainer.compute_rashomon_set(test, domain_map_fn=domain_map_fn)
    break

print("TRAINING WITH SMOOTHING CONSTRAINTS: ") 

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:2], val_tasks[1:2], test_tasks[1:2])
):
    smooth_trainer.train(train, val, batch_size=256)
    smooth_trainer.test(test_tasks[0 : i + 2])
    # if i < len(train_tasks) - 2:
    #     interval_trainer.compute_rashomon_set(test, domain_map_fn=domain_map_fn)
    break
    

### Class Incremental Learning

In [ ]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model()

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

In [ ]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=300,
    min_acc_limit=0.8,
    primal_learning_rate=0.5,
    paradigm="CIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

In [ ]:
import copy

smooth_trainer = SmoothTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=300,
    min_acc_limit=0.8,
    primal_learning_rate=0.5,
    paradigm="CIL",
)

#smooth_trainer.density_strategy == "interval_init"
#smooth_trainer.pre_computed_bound = interval_trainer.bounds[-1]

smooth_trainer.density_strategy = "fim_init"

#smooth_trainer.smooth_steps = 10
#smooth_trainer.density_strategy = "sgd_opt"


smooth_trainer.compute_rashomon_set(test_tasks[0])

#st_safe = copy.deepcopy(smooth_trainer)

In [ ]:


for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:])
):
    interval_trainer.train(train, val, batch_size=256)
    interval_trainer.test(test_tasks[0 : i + 2])
    break
    #if i < len(train_tasks) - 2:
    #    interval_trainer.compute_rashomon_set(test)

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:])
):
    smooth_trainer.train(train, val, batch_size=256)
    smooth_trainer.test(test_tasks[0 : i + 2])
    break
    #if i < len(train_tasks) - 2:
    #    interval_trainer.compute_rashomon_set(test)

### Class Incremental Learning + Regulariser

In [ ]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model()

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
regulariser = UnbiasRegulariser(lmbd=0.01)

trainer = SimpleTrainer(model)
trainer.train(
    train_tasks[0], val_tasks[0], epochs=3, batch_size=256, regulariser=regulariser
)
trainer.test(test_tasks[0:1])

In [ ]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=300,
    min_acc_limit=0.8,
    primal_learning_rate=0.5,
    paradigm="CIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:])
):
    interval_trainer.train(train, val, batch_size=256, regulariser=regulariser)
    interval_trainer.test(test_tasks[0 : i + 2])
    break
    #if i < len(train_tasks) - 2:
    #    interval_trainer.compute_rashomon_set(test)